# TensorNEAT 0.2.0 quickstart

TensorNEAT is a NEAT / HyperNEAT neuroevolution library whose compute layer is the **EvoX Compiler (`evoxc`)** — there is no JAX and no PyTorch anywhere in `src/`.
The NEAT idea is unchanged: irregular, per-individual network topologies are *tensorized* into fixed-shape arrays — `node_keys (P, max_nodes)`, `conn_keys (P, max_conns, key_width)` and their attribute arrays — so a whole population of differently-shaped networks is evaluated in one batch pass.
Every array is **batch-first**: the population sits on axis 0, and every shape is static, so a shape change means a rebuild rather than a reshape.
A kernel is an `evoxc` graph: it is built once (`@evoxc.defn` → `evoxc.build`) into an executable cached per shape signature, then executed on a switchable backend (`"numpy"` reference interpreter by default; `"iree"` and `"tvm"` behind extras, `"xla"` behind a user-supplied PJRT plugin).

This notebook runs top to bottom on the numpy backend, needs no optional dependency, and ends with a solved 2-input XOR.

In [1]:
import tensorneat

print(tensorneat.__version__)

0.2.0


## 1. Build the pieces: genome, algorithm, problem

- `genome.DefaultGenome` — the genotype: genes (node/connection attributes, mutation, crossover) plus the static geometry `max_nodes` / `max_conns` that makes tensorization possible.
- `algorithm.NEAT` — population size, speciation settings, and the `ask` / `tell` loop over that genome.
- `problem.XOR` — the fitness function. Fitness is **maximised** everywhere, so a function-fitting target is a negative error bound: `fitness_target=-1e-3` means "mean squared error at or below 1e-3".

`Pipeline` binds the three together and owns the compile site and the logging.

In [2]:
from tensorneat import Pipeline, algorithm, genome, problem
from tensorneat.common import ACT, to_numpy

neat_genome = genome.DefaultGenome(
    num_inputs=2,
    num_outputs=1,
    max_nodes=8,
    output_transform=ACT.sigmoid,  # bound the read-out so the XOR target is reachable
)
neat = algorithm.NEAT(
    pop_size=100,
    species_size=10,
    survival_threshold=0.01,
    genome=neat_genome,
)
xor = problem.XOR()

pipeline = Pipeline(
    algorithm=neat,
    problem=xor,
    seed=0,                # (seed, config, shape) reproduces a run exactly
    fitness_target=-1e-3,  # fitness is maximised: MSE at or below 1e-3
    generation_limit=100,
    backend="numpy",       # reference interpreter: the default, and the fastest CPU backend today
    is_detailed=False,     # keep this notebook's output small; auto_run logs per generation when True
)
print("backend:", pipeline.backend, "| population:", pipeline.pop_size)

backend: numpy | population: 100


## 2. `setup()` builds the state; the graph is compiled once, on the first `step`

`setup()` builds the initial state — the population tensors, the `evoxc` RNG key and the speciation bookkeeping.
The generation graph is traced, lowered and compiled **lazily, on the first `step`**, and then cached under `(pop_size, max_nodes, max_conns, max_in_degree, max_depth, key_span)`.
That is why `pipeline.builds` still reads `0` below; section 3 shows it going to `1` and staying there, and a different shape is a second cache entry rather than a silent reuse.

In [3]:
state = pipeline.setup()
print("builds right after setup():", pipeline.builds, "| shape signature:", pipeline.shape_signature)
print("population payload:", type(state.pop).__name__)
print("  field shapes:", [tuple(field.shape) for field in state.pop])
print("  generation counter:", to_numpy(state.generation))

builds right after setup(): 0 | shape signature: None
population payload: GenomeTensors
  field shapes: [(100, 8), (100, 8, 4), (100, 100, 2), (100, 100, 1)]
  generation counter: 0.0


## 3. One generation at a time

`pipeline.step(state)` advances exactly one generation — `split randkey → ask → transform → problem.evaluate → tell`, one graph run — and returns `(new_state, evaluated_population, fitnesses)`.
Note where the fitness array is: **last**. Generation counters and tensor leaves come back as `evoxc` tensors, so `tensorneat.common.to_numpy` is the host accessor used for logging.

In [4]:
import numpy as np

for _ in range(3):
    state, evaluated_pop, fitnesses = pipeline.step(state)
    valid = fitnesses[~np.isinf(fitnesses)]  # a NaN-poisoned genome is scored -inf
    print(
        f"generation {int(to_numpy(state.generation))}: "
        f"max={valid.max():+.4f} mean={valid.mean():+.4f} "
        f"valid={valid.size}/{fitnesses.size} builds={pipeline.builds}"
    )

print("compile paid on the first step:", round(pipeline.build_seconds, 3), "s",
      "| shape signature:", pipeline.shape_signature)

generation 1: max=-0.2508 mean=-0.3160 valid=100/100 builds=1
generation 2: max=-0.2501 mean=-0.2557 valid=100/100 builds=1
generation 3: max=-0.2500 mean=-0.2557 valid=94/100 builds=1
compile paid on the first step: 0.299 s | shape signature: (100, 8, 100, 4, 16, 108)


## 4. `auto_run()` to the fitness target

`auto_run(state)` keeps calling `step` until `fitness_target` is reached or `generation_limit` is hit, and returns `(state, best)` with the best genome seen so far.

In [5]:
state, best = pipeline.auto_run(state)
print("generations:", int(to_numpy(state.generation)))
print("best fitness:", round(pipeline.best_fitness, 6))
print("target reached:", pipeline.best_fitness >= pipeline.fitness_target)

generations: 81
best fitness: -0.000934
target reached: True


## 5. `show(state, best)` — what the winner actually computes

In [6]:
pipeline.show(state, best)

input: [0. 0.], target: [0.], predict: [0.04345603]
input: [0. 1.], target: [1.], predict: [0.9915045]
input: [1. 0.], target: [1.], predict: [0.96281934]
input: [1. 1.], target: [0.], predict: [0.01981727]
loss: 0.000933931558392942


## 6. `best` is a batch-first, one-individual payload

`best` is a `GenomeTensors` — the same four-array payload as `state.pop`, but with a population axis of 1.
`Pipeline.show(state, best)` and `genome.network_dict(state, best)` therefore each take it as **one** argument.
The pre-0.2.0 docs call `network_dict(state, nodes, conns)` / `network_dict(state, *best)`; that is broken API now, because the payload is a single object, not loose rows.

In [7]:
print("best:", [tuple(field.shape) for field in best])

network = neat_genome.network_dict(state, best)
print("nodes:", sorted(network["nodes"]))
print("connections:", sorted(network["conns"]))
print("evaluation layers:", network["topo_layers"])
print("nodes reaching an output:", sorted(network["useful_nodes"]))

best: [(1, 8), (1, 8, 4), (1, 100, 2), (1, 100, 1)]
nodes: [0, 1, 7, 8, 9]
connections: [(0, 7), (0, 8), (0, 9), (1, 7), (1, 8), (1, 9), (8, 7), (9, 7), (9, 8)]
evaluation layers: [[0, 1], [9], [8], [7]]
nodes reaching an output: [0, 1, 7, 8, 9]


In [8]:
try:
    neat_genome.network_dict(state, *best)
except TypeError as error:
    print("network_dict(state, *best) ->", type(error).__name__, "-", error)

network_dict(state, *best) -> TypeError - DefaultGenome.network_dict() takes from 3 to 4 positional arguments but 6 were given


## 7. Backends: `backend=` selects one; this notebook stays on `numpy`
`backend=` is forwarded verbatim to `evoxc.build`, so the same graph runs on the reference interpreter or on a compiled backend without changing a line of the program:
- `"numpy"` (the default, used above) — the reference interpreter; on CPU it is the fastest backend TensorNEAT ships today, and the only one this notebook exercises.
- `"iree"` — install the extra from a checkout with `uv sync --extra iree`, or without uv with `pip install "evoxc[iree] @ git+https://github.com/EMI-Group/evoxc@v0.1.1"`; plain `pip install "evoxc[iree]"` is not supported, because PyPI only ships `evoxc` 0.1.0 while `pyproject.toml` resolves the git tag `v0.1.1` through `[tool.uv.sources]`.
  Each static shape then pays one `iree-compile` (7.0–10.4 s per generation graph, 8.4–8.8 s of it at pop 100), and the compiled runs are **slower** per generation than numpy on this box: 9.3× at pop 100, 93.9× at pop 500, 102.7× at pop 1000 (median ratios; `benchmarks/BACKENDS.md`).
- `"xla"` — there is **no pip extra and no installable dependency**: the evoxc xla adapter drives a user-supplied PJRT C API plugin `.so`, found through `plugin_path` in `compile_options` or the `EVOXC_PJRT_PLUGIN` environment variable, and nothing provides one on this box.
- `"tvm"` — an `evoxc[tvm]` extra exists; this project's measurements did not attempt it.
An unsupported backend, dtype or shape raises the backend's own error; nothing silently falls back to another backend.
The measurements, the protocol and the per-gate fix record live in `benchmarks/BACKENDS.md`; the numpy-vs-frozen-legacy performance record is in `benchmarks/RESULTS.md`.
The `iree` leg is *not* run in this notebook: the cell below only reports whether the extra is installed, because iree pays a 7–10 s compile per shape and is slower per generation here — the numpy leg above is the runtime this notebook measures.

In [9]:
import importlib.util

print("iree installed here:", importlib.util.find_spec("iree") is not None)
print("backend this notebook ran on:", pipeline.backend)

iree installed here: False
backend this notebook ran on: numpy


## 8. A kernel really is an `evoxc` graph

`evoxc.build` turns a decorated Python function into an `evoxc.pipeline.Executable`; `evoxc.run` executes it on host arrays. A small standalone graph first:

In [10]:
import evoxc
from evoxc import numpy as enp


@evoxc.defn
def row_total(x):
    return enp.sum(x, axis=1)


mini = evoxc.build(row_total, evoxc.TensorSpec((4, 3), evoxc.float32))
print("built:", type(mini).__module__ + "." + type(mini).__name__,
      "| backend:", mini.backend_executable.backend_name,
      "| entry points:", mini.backend_executable.functions)

host_input = np.ones((4, 3), dtype=np.float32)
print("result:", to_numpy(evoxc.run(mini, evoxc.from_numpy(host_input))))

built: evoxc.pipeline.Executable | backend: numpy | entry points: ('main',)
result: [3. 3. 3. 3.]


The pipeline's generation kernel is the same kind of object: the **first `step()`** built one `evoxc.pipeline.Executable` and every generation since has reused it (that is the `builds=0` → `builds=1` transition in section 3); `setup()` alone builds nothing, it only creates the state.
The accessor below is private and is shown only to make the mechanics visible; `pipeline.builds`, `pipeline.build_seconds` and `pipeline.shape_signature` are the supported counters.

In [11]:
executable = pipeline._executable_for(state)
print(type(executable).__module__ + "." + type(executable).__name__)
print("backend:", executable.backend_executable.backend_name)
print("entry points:", executable.backend_executable.functions)
print("graph inputs:", len(executable.signature.input_specs),
      "| graph outputs:", len(executable.signature.output_specs))
print("executables cached by this pipeline:", len(pipeline._executables))

evoxc.pipeline.Executable
backend: numpy
entry points: ('main',)
graph inputs: 16 | graph outputs: 18
executables cached by this pipeline: 1


## Where to go next

- `examples/` — one runnable script per problem family: `func_fit/xor.py`, `func_fit/xor3d.py`, `func_fit/custom_func_fit.py` (Pagie polynomial through `CustomFuncFit`), `func_fit/xor_recurrent.py` (`RecurrentGenome`), both HyperNEAT variants, the three simulator-free RL environments (`rl/cartpole.py` for `CartPole`, `rl/pendulum.py` for `Pendulum` with its continuous torque, `rl/mountaincar.py` for `MountainCarContinuous` with its continuous force), and host-only introspection in `interpret_visualize/`.
- The `CONTEXT.md` chain — the repo-root `CONTEXT.md` → `src/tensorneat/CONTEXT.md` → the subtree you need (`genome/`, `algorithm/neat/`, `problem/`, `common/`); each one states the intent, API surface, constraints and routing table of its directory.
- `tutorials/legacy_jax/` — the pre-0.2.0 JAX notebooks, archived for concepts only; none of that code runs against this library.
- `benchmarks/BACKENDS.md` and `benchmarks/RESULTS.md` — backend numbers and the old-vs-new performance record.